# FreshMart Lab 1: Exploratory Data Analysis
**Microsoft Fabric Data Science**

บทบาท: **Analyst** — สำรวจของเสียและพฤติกรรม Churn  
รันทีละเซลล์ จดสามข้อสังเกตสั้น ๆ ก่อน Lab 2 ไม่ต้องจำสูตรสถิติ


In [ ]:
from pathlib import Path
import pandas as pd

def _first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None

def load_csv(file_name: str) -> pd.DataFrame:
    found = _first_existing([
        f"/lakehouse/default/Files/raw/{file_name}",
        f"Files/raw/{file_name}",
        f"../data/{file_name}",
        f"labs/data/{file_name}",
        file_name,
    ])
    if found is None:
        raise FileNotFoundError(f"Cannot find {file_name}. Upload it to Files/raw or place it under labs/data.")
    print(f"Loaded CSV: {found}")
    return pd.read_csv(found)

def load_table_or_csv(table_name: str, file_name: str) -> pd.DataFrame:
    try:
        frame = spark.read.table(table_name).toPandas()
        print(f"Loaded Spark table {table_name}: {len(frame):,} rows")
        return frame
    except Exception as exc:
        print(f"Spark table '{table_name}' unavailable ({exc}). Falling back to CSV.")
        return load_csv(file_name)


### ขั้นตอนที่ 1: โหลดข้อมูลธุรกรรม


In [ ]:
df = load_table_or_csv("bronze_transactions", "freshmart_transactions.csv")
print(f"Pandas DataFrame shape: {df.shape}")
df.head()


### ขั้นตอนที่ 2: Data Quality


In [ ]:
print("=== DataFrame Info ===")
df.info()
print("\n=== Missing Values Count ===")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"DiscountRate missing rate: {df['DiscountRate'].isna().mean():.2%}")


### ขั้นตอนที่ 3: Descriptive Statistics


In [ ]:
df[["UnitsSold", "UnitPrice", "DiscountRate", "SalesAmount", "WasteUnits", "WasteCost"]].describe()


### ขั้นตอนที่ 4: Histogram และ Box Plot


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["UnitsSold"], bins=15, kde=True, color="#00D2B4", ax=axes[0])
axes[0].set_title("Distribution of Units Sold")
sns.histplot(df["WasteUnits"], bins=10, kde=True, color="#F43F5E", ax=axes[1])
axes[1].set_title("Distribution of Waste Units (right-skewed)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="StoreType", y="WasteUnits", hue="StoreType", palette="Set2", legend=False)
plt.title("Waste Units by Store Type")
plt.show()


### ขั้นตอนที่ 5: Correlation Heatmap


In [ ]:
numeric_cols = ["UnitsSold", "UnitPrice", "DiscountRate", "SalesAmount", "WasteUnits", "WasteCost", "IsWeekend"]
corr_matrix = df[numeric_cols].corr(numeric_only=True)
print(corr_matrix.round(2))

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, linewidths=0.5)
plt.title("FreshMart Feature Correlation Matrix")
plt.show()


### ขั้นตอนที่ 6: Customer Churn EDA


In [ ]:
df_cust = load_table_or_csv("bronze_customers", "freshmart_customers.csv")
print(f"Total Customers: {len(df_cust):,}")
print(df_cust["Churn"].value_counts(normalize=True).rename("rate"))
print(f"Age missing: {df_cust['Age'].isna().sum()} ({df_cust['Age'].isna().mean():.2%})")

plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=df_cust,
    x="RecencyDays",
    y="ComplaintCount",
    hue="Churn",
    palette={0: "#00D2B4", 1: "#F43F5E"},
    alpha=0.7,
)
plt.title("Customer Churn Pattern: Recency vs Complaint Count")
plt.show()


In [ ]:
if len(df) != 3000:
    raise AssertionError(f"คาดว่าธุรกรรม 3,000 แถว แต่ได้ {len(df):,}")
if len(df_cust) != 1500:
    raise AssertionError(f"คาดว่าสมาชิก 1,500 แถว แต่ได้ {len(df_cust):,}")
if df["DiscountRate"].isna().sum() == 0:
    raise AssertionError("ชุดนี้ควรมี DiscountRate ว่าง เพื่อฝึกจัดการค่าว่างใน Lab 2")
if df_cust["Age"].isna().sum() == 0:
    raise AssertionError("ชุดนี้ควรมี Age ว่าง เพื่อฝึกเติมค่าใน Lab 2")
print("Lab 1 verification passed")
